In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_DIR = Path.cwd()
FILE_500 = DATA_DIR / "PokemonGO_500final.csv"
FILE_2000 = DATA_DIR / "PokemonGO_Main_2000.csv"
OUTPUT = DATA_DIR / "PokemonGO_Merged_2500_V1_2.csv"
LABEL_COLS = [f"A{i:02d}" for i in range(1, 9)] + ["G", "B01"]

for f in (FILE_500, FILE_2000):
    if not f.is_file():
        raise FileNotFoundError(f"입력 파일을 찾을 수 없습니다: {f}")
print("500건 입력:", FILE_500.name)
print("2000건 입력:", FILE_2000.name)

500건 입력: PokemonGO_Reliability_500_사람A.csv
2000건 입력: PokemonGO_Main_2000.csv


In [4]:
def read_csv_text(path):
    # ID 및 코딩값의 의도하지 않은 자동 숫자 변환을 방지합니다.
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)

df500 = read_csv_text(FILE_500)
df2000 = read_csv_text(FILE_2000)
assert len(df500) == 500, f"500건 입력 행 수 오류: {len(df500)}"
assert len(df2000) == 2000, f"2000건 입력 행 수 오류: {len(df2000)}"
assert list(df500.columns) == list(df2000.columns), "열 구성이 서로 다릅니다. 입력 파일과 코딩북 버전을 확인하세요."
for name, df in [("사람 A 500건", df500), ("본 분석 2000건", df2000)]:
    assert not df["ID"].eq("").any(), f"{name}: 빈 ID가 있습니다."
    assert not df["ID"].duplicated().any(), f"{name}: 중복 ID가 있습니다."
    for col in LABEL_COLS:
        assert col in df.columns, f"{name}: {col} 열이 없습니다."
    for col in [f"A{i:02d}" for i in range(1, 9)] + ["G"]:
        invalid = set(df[col].unique()) - {"-1", "0", "1", "9", "U"}
        assert not invalid, f"{name} {col}: 허용되지 않은 값 {invalid}"
    assert set(df["B01"].unique()) <= {"0", "1", "U"}, f"{name}: B01 코드 검증 실패"
assert set(df500["ID"]).isdisjoint(set(df2000["ID"])), "두 표본에 중복되는 ID가 있습니다!"
print("구조 검증 통과: 두 파일 모두",len(df500.columns),"개 열; ID 중복 없음")

구조 검증 통과: 두 파일 모두 17 개 열; ID 중복 없음


In [5]:
a = df500.copy()
b = df2000.copy()
a.insert(0, "자료구분", "사람A_500")
b.insert(0, "자료구분", "본분석_2000")
merged = pd.concat([a, b], axis=0, ignore_index=True)
merged.insert(0, "통합번호", [f"P{i:04d}" for i in range(1, len(merged) + 1)])
assert len(merged) == 2500
assert merged["ID"].nunique() == 2500
assert (merged.loc[:499, df500.columns] .reset_index(drop=True) == df500).all().all(), "500건 원본 값이 변경되었습니다."
assert (merged.loc[500:, df2000.columns].reset_index(drop=True) == df2000).all().all(), "2000건 원본 값이 변경되었습니다."
print("완료:", merged.shape[0], "행 ×", merged.shape[1], "열")

완료: 2500 행 × 19 열


In [6]:
star_table = pd.crosstab(merged["별점_숫자"], merged["자료구분"]).sort_index()
star_table["합계"] = star_table.sum(axis=1)
display(star_table)
assert star_table["합계"].to_dict() == {str(i):500 for i in range(1,6)}, "별점별 표본 수가 예상과 다릅니다."
u_counts = (merged[LABEL_COLS] == "U").sum().rename("U_건수")
display(u_counts.to_frame())
print("U가 하나 이상 있는 리뷰:", (merged[LABEL_COLS] == "U").any(axis=1).sum())

자료구분,본분석_2000,사람A_500,합계
별점_숫자,,,
1,400,100,500
2,400,100,500
3,400,100,500
4,400,100,500
5,400,100,500


,U_건수
A01,86
A02,14
A03,23
A04,41
A05,1
A06,18
A07,109
A08,1
G,6
B01,0


U가 하나 이상 있는 리뷰: 267


In [7]:
merged.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print("파일 저장 완료:", OUTPUT.resolve())
print("저장된 행 수:", len(read_csv_text(OUTPUT)))

파일 저장 완료: D:\vscode_project\포켓몬고\PokemonGO_Merged_2500_V1_2.csv
저장된 행 수: 2500
